# Chapter 7: Energy-based models, score matching and Langevin sampling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHHamdan/rose-explorer/blob/main/notebooks/07_energy_based_models_and_score_matching.ipynb)

An energy-based model (EBM) writes a density as $e^{-E(\mathbf{x})}/Z$ and leaves the normaliser $Z$ alone. This chapter follows the consequences: the maximum-likelihood gradient needs samples from the model (the negative phase); the *score* $\nabla_{\mathbf{x}}\log p$ does not need $Z$ at all, and can be fitted by score matching (Hyvärinen's integration by parts), by sliced score matching, or by denoising score matching, whose minimiser is the score of a *smoothed* density; a score is enough to sample by Langevin dynamics. The chapter ends where diffusion models begin: denoising score matching at many noise levels is the DDPM loss up to a weighting, and the score of a posterior is the prior score plus a likelihood gradient.

**Paper sections:** main §3.4–3.5 and §5.5; Supplement S1.7–S1.8 and S4.5.  **Prerequisites:** Chapters 1 and 2 (KL divergence, maximum likelihood).

**How to read this chapter.** Each equation has three layers. *In plain words* is for a first encounter. *Refresher* states the assumptions and sketches the derivation for readers who have seen the material. *Verify* checks the identity in code; run every cell (Runtime > Run all) and the last cell reports whether every check passed. Equation ids such as `EQ-M-007` refer to `data/equation_registry.json` in the repository, which maps each id to its line in the paper.

**Found an error?** Open an issue quoting the equation id, or fix the entry in `notebooks/src/07_energy_based_models_and_score_matching.py` and run `python tools/check_notebooks.py`.

<details><summary>Notation used throughout</summary>

| Symbol | Meaning |
|---|---|
| $\mathbf{x}$, $\mathbf{z}$ | an observed data point; a latent (unobserved) variable |
| $p_{\mathrm{data}}$, $p_\theta$ | the data distribution; the model's distribution with parameters $\theta$ |
| $\mathbb{E}_p[f]$ | the average of $f$ when its argument is drawn from $p$ |
| $D_{\mathrm{KL}}(p\,\Vert\,q)$ | Kullback--Leibler divergence, $\mathbb{E}_p[\log p - \log q]$ |
| $\mathcal{N}(\mu,\Sigma)$ | a Gaussian with mean $\mu$ and covariance $\Sigma$ |
| $\epsilon$ | standard Gaussian noise, $\epsilon\sim\mathcal{N}(0,I)$ |
| $\bar\alpha_t$, $\beta_t$ | diffusion noise-schedule quantities, $\bar\alpha_t=\prod_{s\le t}(1-\beta_s)$ |
| $\mathbf{s}_\theta(\mathbf{x})$ | a score network, an estimate of $\nabla_{\mathbf{x}}\log p(\mathbf{x})$ |
| $\mathbf{c}$ | a conditioning signal (a class label, a text prompt) |

</details>

In [ ]:
# Setup: CPU only, small sizes, fixed seeds. Everything here is pre-installed on Google Colab.
import math, os, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = ""          # the checks are meant to run on a CPU
import numpy as np
import scipy, scipy.stats, scipy.integrate, scipy.linalg, scipy.optimize, scipy.special
import sympy as sp
import matplotlib
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)
rng = np.random.default_rng(0)
torch.manual_seed(0)

RESULTS = []   # (name, passed, detail) for every check in this notebook

def check(name, ok, detail=""):
    """Record one verification result and print it. The last cell fails if any check failed."""
    ok = bool(ok)
    RESULTS.append((name, ok, str(detail)))
    print(("PASS  " if ok else "FAIL  ") + name + ("  |  " + str(detail) if detail else ""))
    return ok

def close(a, b, tol, what=""):
    """|a - b| <= tol elementwise; returns (ok, message) for check()."""
    err = float(np.max(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))))
    return err <= tol, f"{what}|diff| = {err:.3e} (tolerance {tol:.1e})"

print("numpy", np.__version__, "| scipy", scipy.__version__, "| sympy", sp.__version__, "| torch", torch.__version__)

## In this chapter

1. The energy-based density
2. Maximum-likelihood gradient: positive and negative phase
3. The score of an EBM needs no normaliser
4. Three concrete energies: RBM, joint energy model, product of experts
5. Hyvärinen score matching: the Fisher divergence by integration by parts
6. Sliced score matching: an unbiased trace
7. Denoising score matching and its minimiser
8. Langevin dynamics
9. A deep EBM in practice: the IGEBM loss and decoupled sampler noise
10. Annealed Langevin, the $\sigma^2$ weighting and the DDPM loss
11. Recovering the score from a noise-prediction network
12. The posterior score: conditioning without retraining

**See it move.** The *Score and Langevin* tab of the ROSE explorer (`app/` in the repository, also on the project's GitHub Pages site) animates this chapter's main result.

## Energy-based models

Three statements define the family: the density, the gradient of its log-likelihood, and the fact that its score needs no normaliser. Everything else in the chapter is a way of living with the intractable $Z_\theta$.

### 7.1 The energy-based density

*Paper:* `EQ-M-018` main paper, `03_foundations.tex:124-126`, (inline) EBM density; `EQ-S1-029` Supplement S1, `03_foundations.tex:306-315`, `eq:ebm`; `EQ-S4-033` Supplement S4, `06e_ebm_score.tex:22-44`, `eq:ebm_def, eq:ebm_grad_local, EBM score`

$$
p_\theta(\mathbf{x})=\frac{\exp(-E_\theta(\mathbf{x}))}{Z_\theta},\qquad Z_\theta=\int\exp(-E_\theta(\mathbf{x}))\,d\mathbf{x}
$$

**In plain words.** $E_\theta(\mathbf{x})$ is any function (any neural network) that returns one number, the *energy*, for an input $\mathbf{x}$. Low energy means 'likely'. Turning energies into probabilities takes two steps: exponentiate the negative energy, so that the result is positive and lower energy gives a larger value, then divide by the total $Z_\theta$ so that the probabilities sum (or integrate) to one. $Z_\theta$ is called the partition function. For a network on images it is an integral over every possible image, which cannot be computed; that single fact drives the whole chapter.

**Refresher.** This is the Boltzmann (Gibbs) form of statistical physics, at unit temperature. It is fully general: any strictly positive density is an EBM with $E=-\log p+\text{const}$, so the family imposes no restriction on $p$, only on what can be *computed*. $Z_\theta$ depends on $\theta$ but not on $\mathbf{x}$, which is why $\log p_\theta$ is known only up to an additive constant and why ratios $p_\theta(\mathbf{x})/p_\theta(\mathbf{x}')$ and the score are tractable while $\log p_\theta$ itself is not. The supplement (S4.5) lists the consequences: no bits/dim, so EBMs are not in the paper's likelihood comparisons, and no exact sample, only MCMC. The check below uses a *discrete* state space where $Z_\theta$ is a finite sum, so that every quantity is exact; the same discrete model is reused for the gradient identity.

**Verify.**

In [ ]:
# A discrete EBM on S = 12 states with a linear energy E_theta(x) = theta . phi(x): Z is a finite sum.
S, P = 12, 3
phi = rng.normal(size=(S, P))                      # features phi(x) for each of the S states
theta = rng.normal(size=P)
E = phi @ theta                                    # energies E_theta(x)
logZ = scipy.special.logsumexp(-E)
p = np.exp(-E - logZ)
check("p_theta sums to one", abs(p.sum() - 1) < 1e-12, f"sum = {p.sum():.15f}")
check("p_theta is positive everywhere", np.all(p > 0))
# lower energy <=> higher probability (strictly monotone)
order_E, order_p = np.argsort(E), np.argsort(-p)
check("ordering by energy (ascending) = ordering by probability (descending)", np.array_equal(order_E, order_p))
# log-ratios never need Z
i, j = 2, 7
check("log p(x_i)/p(x_j) = E(x_j) - E(x_i)", *close(np.log(p[i] / p[j]), E[j] - E[i], 1e-12))
# adding a constant to the energy changes nothing (it cancels in Z)
E2 = E + 5.0
p2 = np.exp(-E2 - scipy.special.logsumexp(-E2))
check("p_theta is invariant to a constant shift of E", *close(p, p2, 1e-14))

### 7.2 Maximum-likelihood gradient: positive and negative phase

*Paper:* `EQ-M-019` main paper, `03_foundations.tex:128-132`, `eq:ebm_grad`; `EQ-S1-030` Supplement S1, `03_foundations.tex:316-323`, `eq:ebm_grad`

$$
\nabla_\theta\,\mathbb{E}_{\mathbf{x}\sim p_{\mathrm{data}}}\big[\log p_\theta(\mathbf{x})\big]=-\underbrace{\mathbb{E}_{\mathbf{x}\sim p_{\mathrm{data}}}\big[\nabla_\theta E_\theta(\mathbf{x})\big]}_{\text{positive phase}}+\underbrace{\mathbb{E}_{\mathbf{x}\sim p_\theta}\big[\nabla_\theta E_\theta(\mathbf{x})\big]}_{\text{negative phase}},\qquad \nabla_\theta\log Z_\theta=-\mathbb{E}_{p_\theta}\big[\nabla_\theta E_\theta\big]
$$

**In plain words.** To make the data more likely, push the energy of real data points *down* (the first term: the gradient of the energy at data points, with a minus sign) and push the energy of the model's own samples *up* (the second term). The two forces balance exactly when the model's samples look like the data. The catch is the second term: it averages over samples from $p_\theta$, and drawing those requires Markov chain Monte Carlo (MCMC), because $Z_\theta$ is unknown.

**Refresher.** Write $\log p_\theta(\mathbf{x})=-E_\theta(\mathbf{x})-\log Z_\theta$ and differentiate under the integral: $\nabla_\theta\log Z_\theta=Z_\theta^{-1}\int\nabla_\theta e^{-E_\theta}=-\int p_\theta\nabla_\theta E_\theta=-\mathbb{E}_{p_\theta}[\nabla_\theta E_\theta]$. Exchanging $\nabla_\theta$ and $\int$ needs $E_\theta$ differentiable in $\theta$ with a dominating integrable bound on $|\nabla_\theta e^{-E_\theta}|$. In practice the negative-phase expectation is replaced by short MCMC chains: contrastive divergence (Hinton, 2002) starts them at data, persistent chains (Tieleman, 2008) carry them across updates, deep EBMs use short-run Langevin chains from noise or a replay buffer (Du and Mordatch, 2019; Nijkamp et al., 2019). Because the chains do not mix, the estimate is biased (Song and Kingma, 2021), which is why the paper's benchmark omits a trained EBM (main §8) and why the score-matching objectives below avoid $Z_\theta$ altogether. The check ports `ebm_loglik_gradient_positive_negative_phase` from `verification/verify_equations.py`: a finite-difference gradient of the exact log-likelihood against the two-phase formula.

**Verify.**

In [ ]:
# Discrete EBM (Z exact); compare a finite-difference gradient of E_data[log p_theta] with the formula.
rng2 = np.random.default_rng(11)
phi = rng2.normal(size=(12, 3))
p_data = rng2.dirichlet(np.ones(12))
th = rng2.normal(size=3)

def avg_loglik(t):
    logp = -phi @ t - scipy.special.logsumexp(-phi @ t)
    return p_data @ logp

eps_fd = 1e-6
fd = np.array([(avg_loglik(th + eps_fd * np.eye(3)[k]) - avg_loglik(th - eps_fd * np.eye(3)[k])) / (2 * eps_fd)
               for k in range(3)])
p_model = np.exp(-phi @ th - scipy.special.logsumexp(-phi @ th))
grad_E = phi                                         # d E_theta(x)/d theta = phi(x) for a linear energy
formula = -p_data @ grad_E + p_model @ grad_E        # -E_data[dE] + E_model[dE]
check("finite-difference gradient = -E_data[grad E] + E_model[grad E]", *close(fd, formula, 1e-8))
# the normaliser identity on its own
fd_logZ = np.array([(scipy.special.logsumexp(-phi @ (th + eps_fd * np.eye(3)[k]))
                     - scipy.special.logsumexp(-phi @ (th - eps_fd * np.eye(3)[k]))) / (2 * eps_fd) for k in range(3)])
check("grad log Z = -E_model[grad E]", *close(fd_logZ, -(p_model @ grad_E), 1e-8))
# at the maximum-likelihood solution the two phases balance: fit theta, then check the moments match
res = scipy.optimize.minimize(lambda t: -avg_loglik(t), th)
p_fit = np.exp(-phi @ res.x - scipy.special.logsumexp(-phi @ res.x))
check("at the MLE, E_data[grad E] = E_model[grad E] (moment matching)",
      *close(p_data @ grad_E, p_fit @ grad_E, 1e-5))

### 7.3 The score of an EBM needs no normaliser

*Paper:* `EQ-M-020` main paper, `03_foundations.tex:136-138`, (inline) score of an EBM; `EQ-S1-031` Supplement S1, `03_foundations.tex:337-343`, `eq:score`

$$
\mathbf{s}_\theta(\mathbf{x})\;:=\;\nabla_{\mathbf{x}}\log p_\theta(\mathbf{x})\;=\;-\nabla_{\mathbf{x}}E_\theta(\mathbf{x})
$$

**In plain words.** The *score* is the gradient of the log-density with respect to the input: at each point it is an arrow pointing towards higher probability. For an EBM, $\log p_\theta=-E_\theta-\log Z_\theta$ and $\log Z_\theta$ is a constant that does not depend on $\mathbf{x}$, so it disappears when we differentiate. The score is therefore just minus the gradient of the energy, something automatic differentiation gives for free. This is the fact that connects EBMs to score matching and to diffusion models, which learn the score directly.

**Refresher.** 'Score' here is the Stein score (gradient in $\mathbf{x}$), not Fisher's score (gradient in $\theta$). The identity requires $E_\theta$ differentiable in $\mathbf{x}$ and holds on the interior of the support. Two consequences used later: (i) a vector field $\mathbf{s}_\theta$ modelled directly, without an energy, is a 'score model'; it is a conservative field only if it is a gradient, which a free-form network does not guarantee; (ii) every quantity below (score matching, Langevin dynamics, the reverse SDE of Chapter 8) consumes the score and never $Z_\theta$. The check builds a small energy network in torch and compares autograd of $\log p_\theta$ (including an explicit, $\mathbf{x}$-independent $\log Z$ term) with autograd of $-E_\theta$.

**Verify.**

In [ ]:
# A small MLP energy on R^2; the score by autograd of log p (with any constant log Z) equals -grad E.
net = torch.nn.Sequential(torch.nn.Linear(2, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
x = torch.randn(5, 2, requires_grad=True)
E_x = net(x).squeeze(-1)
logZ = torch.tensor(3.14159)                          # any constant: it does not depend on x
logp = -E_x - logZ
score_from_logp = torch.autograd.grad(logp.sum(), x, retain_graph=True)[0]
neg_gradE = -torch.autograd.grad(E_x.sum(), x)[0]
check("illustration: grad_x log p_theta = -grad_x E_theta by autograd (log Z is a constant, so it drops out)", *close(score_from_logp.numpy(), neg_gradE.numpy(), 1e-6))
# analytic case: Gaussian energy E = (x-m)^2/(2v) has score -(x-m)/v
m, v = 1.0, 0.5
xs = np.linspace(-2, 3, 7)
check("Gaussian energy: score = -(x-m)/v (second-order finite differences are exact for a quadratic)",
      *close(-(xs - m) / v, -np.gradient((xs - m) ** 2 / (2 * v), xs, edge_order=2), 1e-9))
# the finite-difference check above is coarse on a grid; the exact statement is symbolic:
xx = sp.symbols("x", real=True)
mm, vv = sp.symbols("m v", positive=True)
check("symbolic: d/dx[-(x-m)^2/(2v)] = -(x-m)/v", sp.simplify(sp.diff(-(xx - mm) ** 2 / (2 * vv), xx) + (xx - mm) / vv) == 0)

### 7.4 Three concrete energies: RBM, joint energy model, product of experts

*Paper:* `EQ-S4-035` Supplement S4, `06e_ebm_score.tex:52-54,64-66,141-142`, (inline) RBM energy; JEM energy; product of experts

$$
E(\mathbf{v},\mathbf{h})=-\mathbf{v}^\top W\mathbf{h}-\mathbf{b}^\top\mathbf{v}-\mathbf{c}^\top\mathbf{h};\qquad E_\theta(\mathbf{x})=-\log\sum_y\exp\big(f_\theta(\mathbf{x})_y\big);\qquad e^{-(E_1+E_2)}\propto p_1\,p_2
$$

**In plain words.** Three examples of what an energy can be. A *restricted Boltzmann machine* scores a pair of binary vectors, visible $\mathbf{v}$ and hidden $\mathbf{h}$, with a bilinear energy; because there are no connections within a layer, the hidden units are independent given the visible ones and vice versa. A *joint energy model* reads a $C$-way classifier's logits $f_\theta(\mathbf{x})$ as $\log p(\mathbf{x},y)$ up to a constant, so the energy of $\mathbf{x}$ alone is minus the log-sum-exp of the logits: the same network classifies and generates. Finally, energies *add*: the density with energy $E_1+E_2$ is the normalised product $p_1p_2/Z$, so two separately trained models can be combined at sampling time without retraining.

**Refresher.** RBM: marginalising $\mathbf{h}$ gives $p(\mathbf{v})\propto e^{\mathbf{b}^\top\mathbf{v}}\prod_j(1+e^{c_j+(W^\top\mathbf{v})_j})$, a tractable unnormalised density, and the conditionals $p(h_j=1\mid\mathbf{v})=\sigma(c_j+(W^\top\mathbf{v})_j)$ make block Gibbs sampling exact in each half-step (Smolensky, 1986; Hinton, 2002). JEM (Grathwohl et al., 2020): $p_\theta(\mathbf{x},y)=e^{f_\theta(\mathbf{x})_y}/Z$, hence $p_\theta(y\mid\mathbf{x})$ is the softmax (the normaliser cancels) and $p_\theta(\mathbf{x})=\sum_y p_\theta(\mathbf{x},y)=e^{\mathrm{LSE}_y f_\theta(\mathbf{x})_y}/Z$. Product of experts (Hinton, 2002): $Z_{12}=\int e^{-E_1-E_2}\neq Z_1Z_2$ in general, so the product is $p_1p_2/\int p_1p_2$, not $p_1p_2$; only the shape is the product. All three are checked on binary or finite state spaces where the sums are exact.

**Verify.**

In [ ]:
# (1) RBM with 3 visible and 2 hidden binary units: exact marginal and conditional from the joint.
import itertools
W = rng.normal(size=(3, 2)); b = rng.normal(size=3); cc = rng.normal(size=2)
V = np.array(list(itertools.product([0, 1], repeat=3)), float)
H = np.array(list(itertools.product([0, 1], repeat=2)), float)
E_joint = -(V @ W @ H.T) - (V @ b)[:, None] - (H @ cc)[None, :]      # shape (8, 4)
joint = np.exp(-E_joint); joint /= joint.sum()
marg_v = joint.sum(1)
free = np.exp(V @ b) * np.prod(1 + np.exp(cc + V @ W), axis=1)      # exp(-free energy), unnormalised
check("RBM: marginal p(v) = exp(-F(v))/Z with F the free energy", *close(marg_v, free / free.sum(), 1e-12))
cond_h1 = (joint[:, H[:, 0] == 1].sum(1)) / marg_v                     # p(h_1 = 1 | v) from the joint
check("RBM: p(h_j=1|v) = sigmoid(c_j + (W^T v)_j)", *close(cond_h1, scipy.special.expit(cc[0] + V @ W[:, 0]), 1e-12))

# (2) JEM: logits f(x)_y on a finite grid of 6 x-states and 4 classes.
f = rng.normal(size=(6, 4))
p_xy = np.exp(f); p_xy /= p_xy.sum()
E_x = -scipy.special.logsumexp(f, axis=1)
p_x = np.exp(-E_x); p_x /= p_x.sum()
check("JEM: p(x) = sum_y p(x,y) = exp(-E(x))/Z with E = -LSE_y f(x)_y", *close(p_x, p_xy.sum(1), 1e-12))
check("JEM: p(y|x) is the softmax of the logits", *close(p_xy / p_xy.sum(1, keepdims=True), scipy.special.softmax(f, 1), 1e-12))

# (3) Product of experts on 10 states: density of E_1+E_2 is p_1 p_2 renormalised.
E1, E2 = rng.normal(size=10), rng.normal(size=10)
p1 = np.exp(-E1) / np.exp(-E1).sum(); p2 = np.exp(-E2) / np.exp(-E2).sum()
p12 = np.exp(-(E1 + E2)); p12 /= p12.sum()
check("product of experts: exp(-(E1+E2))/Z = p1 p2 / sum(p1 p2)", *close(p12, p1 * p2 / (p1 * p2).sum(), 1e-12))
check("...and Z_12 != Z_1 Z_2 in general", abs(np.exp(-(E1 + E2)).sum() - np.exp(-E1).sum() * np.exp(-E2).sum()) > 1e-6)

## Score matching

If the score is all we need, we can fit it directly. Hyvärinen's objective removes the unknown data score by integration by parts; sliced score matching makes the Jacobian trace cheap; denoising score matching removes the trace altogether at the price of fitting a smoothed density.

### 7.5 Hyvärinen score matching: the Fisher divergence by integration by parts

*Paper:* `EQ-M-021` main paper, `03_foundations.tex:138-141`, (inline) Hyvarinen score matching; `EQ-S1-032` Supplement S1, `03_foundations.tex:343-351`, `eq:sm`; `EQ-S4-037` Supplement S4, `06e_ebm_score.tex:249-257`, `eq:sm_local; sliced SM`

$$
\tfrac12\,\mathbb{E}_{p_{\mathrm{data}}}\big\|\mathbf{s}_\theta(\mathbf{x})-\nabla_{\mathbf{x}}\log p_{\mathrm{data}}(\mathbf{x})\big\|^2=\underbrace{\mathbb{E}_{p_{\mathrm{data}}}\Big[\operatorname{Tr}\nabla_{\mathbf{x}}\mathbf{s}_\theta(\mathbf{x})+\tfrac12\|\mathbf{s}_\theta(\mathbf{x})\|^2\Big]}_{\mathcal{L}_{\mathrm{SM}}(\theta)}+\underbrace{\tfrac12\,\mathbb{E}_{p_{\mathrm{data}}}\|\nabla_{\mathbf{x}}\log p_{\mathrm{data}}\|^2}_{\text{constant in }\theta}
$$

**In plain words.** We would like to make the model's arrows $\mathbf{s}_\theta$ match the data's arrows $\nabla\log p_{\mathrm{data}}$, by minimising the average squared difference (the *Fisher divergence*, left). But we do not know the data's arrows. Hyvärinen (2005) showed that the cross term can be rewritten, by integration by parts, into something that involves only the model: the divergence of the model's arrows, $\operatorname{Tr}\nabla_{\mathbf{x}}\mathbf{s}_\theta$, that is, how much the field spreads out. Minimising the right-hand side's first bracket over $\theta$ is the same as minimising the Fisher divergence, because the second term does not depend on $\theta$.

**Refresher.** Expand the square: $\tfrac12\mathbb{E}\|\mathbf{s}_\theta\|^2-\mathbb{E}\langle\mathbf{s}_\theta,\nabla\log p\rangle+\tfrac12\mathbb{E}\|\nabla\log p\|^2$. The cross term is $\int\langle\mathbf{s}_\theta,\nabla p\rangle=-\int p\,\nabla\cdot\mathbf{s}_\theta$ by the divergence theorem, provided the boundary term $p\,\mathbf{s}_\theta\to0$ as $\|\mathbf{x}\|\to\infty$, $p$ is differentiable, and $\mathbf{s}_\theta$, $\nabla\log p$ have finite second moments under $p$; the paper states these conditions in S1 (eq:sm). The main paper's inline statement carries a wording fix in the registry: $\mathcal{L}_{\mathrm{SM}}$ equals the Fisher divergence *up to a constant and with the factor $\tfrac12$*; without the $\tfrac12$ the two differ by a factor 2 in the parameter-dependent part. Cost: $\operatorname{Tr}\nabla\mathbf{s}_\theta$ needs $D$ backward passes, the motivation for the sliced and denoising variants. The check ports `hyvarinen_score_matching_equals_fisher_up_to_constant`: for Gaussian data and a Gaussian-family model score, both sides are polynomials in a standard normal and the expectations are exact in sympy.

**Verify.**

In [ ]:
# Exact symbolic check: data x ~ N(mu, s^2); model score s_theta(x) = -(x - a)/b (its Jacobian trace is -1/b).
def gauss_expect(expr, gens):
    """E[expr] for independent standard normals `gens` (expr polynomial): E[u^k] = (k-1)!! for even k, 0 for odd."""
    poly = sp.Poly(sp.expand(expr), *gens)
    tot = 0
    for mon, coef in poly.terms():
        mval = 1
        for k in mon:
            mval *= 0 if k % 2 else sp.factorial2(k - 1)
        tot += coef * mval
    return sp.simplify(tot)

u, a, mu = sp.symbols("u a mu", real=True)
b, s = sp.symbols("b s", positive=True)
xsym = mu + s * u
model_score = lambda y: -(y - a) / b
fisher_half = gauss_expect(sp.Rational(1, 2) * (model_score(xsym) + (xsym - mu) / s**2) ** 2, [u])
L_SM = gauss_expect(-1 / b + sp.Rational(1, 2) * model_score(xsym) ** 2, [u])      # Tr grad s = -1/b
diff = sp.simplify(fisher_half - L_SM)
check("(1/2) Fisher divergence - L_SM is independent of the model parameters (a, b)",
      not (diff.free_symbols & {a, b}), f"difference = {diff} = (1/2) E||grad log p_data||^2")
check("the constant equals (1/2) E||grad log p_data||^2 = 1/(2 s^2)", sp.simplify(diff - 1 / (2 * s**2)) == 0)
# both objectives are minimised at the true score (a = mu, b = s^2)
sol = sp.solve([sp.diff(L_SM, a), sp.diff(L_SM, b)], [a, b], dict=True)
check("argmin of L_SM is the data score (a = mu, b = s^2)",
      any(sp.simplify(d_[a] - mu) == 0 and sp.simplify(d_[b] - s**2) == 0 for d_ in sol), f"solutions: {sol}")

# Numeric check on a 1-D Gaussian mixture (non-Gaussian data), model score from a different mixture, by quadrature.
def mix_logp_and_score(x, w, m, sd):
    comp = np.array([wi * scipy.stats.norm.pdf(x, mi, si) for wi, mi, si in zip(w, m, sd)])
    p = comp.sum(0)
    dp = np.array([-(x - mi) / si**2 for mi, si in zip(m, sd)])
    return np.log(p), (comp * dp).sum(0) / p

xs = np.linspace(-12, 12, 20001)
logp_d, score_d = mix_logp_and_score(xs, [0.3, 0.7], [-2.0, 1.5], [0.8, 1.2])
logp_m, score_m = mix_logp_and_score(xs, [0.5, 0.5], [-1.0, 2.0], [1.0, 0.7])
p_d = np.exp(logp_d)
fisher = 0.5 * np.trapezoid(p_d * (score_m - score_d) ** 2, xs)
Lsm = np.trapezoid(p_d * (np.gradient(score_m, xs) + 0.5 * score_m**2), xs)
const = 0.5 * np.trapezoid(p_d * score_d**2, xs)
check("Gaussian mixture by quadrature: (1/2) Fisher = L_SM + const", *close(fisher, Lsm + const, 1e-6))

### 7.6 Sliced score matching: an unbiased trace

*Paper:* `EQ-S1-033` Supplement S1, `03_foundations.tex:352-354`, (inline) sliced score matching

$$
\operatorname{Tr}\big(\nabla_{\mathbf{x}}\mathbf{s}_\theta(\mathbf{x})\big)=\mathbb{E}_{\mathbf{v}}\big[\mathbf{v}^\top\nabla_{\mathbf{x}}\mathbf{s}_\theta(\mathbf{x})\,\mathbf{v}\big],\qquad \mathbb{E}[\mathbf{v}\mathbf{v}^\top]=I
$$

**In plain words.** The trace of a $D\times D$ Jacobian needs $D$ backward passes. Instead, pick a random direction $\mathbf{v}$ with $\mathbb{E}[\mathbf{v}\mathbf{v}^\top]=I$ (for instance, each coordinate $\pm1$ with equal probability, or a standard Gaussian), and compute how the field changes along that one direction: $\mathbf{v}^\top J\mathbf{v}$, one backward pass. On average over directions this equals the trace exactly. Sliced score matching (Song et al., 2020) uses this estimator inside Hyvärinen's objective.

**Refresher.** $\mathbb{E}[\mathbf{v}^\top J\mathbf{v}]=\mathbb{E}[\operatorname{Tr}(J\mathbf{v}\mathbf{v}^\top)]=\operatorname{Tr}(J\,\mathbb{E}[\mathbf{v}\mathbf{v}^\top])=\operatorname{Tr}J$: Hutchinson's estimator (1989). The variance differs by distribution: for Rademacher $\mathbf{v}$ it is $2\sum_{i\neq j}J_{ij}^2$ (the diagonal contributes nothing), for Gaussian $\mathbf{v}$ it is $2\|J\|_F^2$ (symmetric $J$); Rademacher is never worse. The same estimator computes the divergence term of the instantaneous change of variables along the probability-flow ODE (Chapter 8), where the paper notes that ODE likelihoods carry both solver and estimator noise. The check ports `hutchinson_and_sliced_trace_unbiased` with a 5-standard-error tolerance and reports the two variances.

**Verify.**

In [ ]:
rng3 = np.random.default_rng(13)
J = rng3.normal(size=(6, 6))
n = 400_000
v_r = rng3.choice([-1.0, 1.0], size=(n, 6))
v_g = rng3.normal(size=(n, 6))
est_r = np.einsum("ni,ij,nj->n", v_r, J, v_r)
est_g = np.einsum("ni,ij,nj->n", v_g, J, v_g)
se_r, se_g = est_r.std() / math.sqrt(n), est_g.std() / math.sqrt(n)
check("Rademacher v: mean of v^T J v = Tr J (within 5 s.e.)", *close(est_r.mean(), np.trace(J), 5 * se_r, f"s.e. {se_r:.4f}; "))
check("Gaussian v: mean of v^T J v = Tr J (within 5 s.e.)", *close(est_g.mean(), np.trace(J), 5 * se_g, f"s.e. {se_g:.4f}; "))
Jsym = 0.5 * (J + J.T)                      # v^T J v = v^T Jsym v; variance formulas hold for the symmetric part
var_r_theory = 2 * (np.sum(Jsym**2) - np.sum(np.diag(Jsym) ** 2))
var_g_theory = 2 * np.sum(Jsym**2)
check("Rademacher variance = 2 sum_{i!=j} J_ij^2 (symmetric part)", *close(est_r.var(), var_r_theory, 0.05 * var_r_theory))
check("Gaussian variance = 2 ||J||_F^2 (symmetric part), larger than Rademacher", est_g.var() > est_r.var() and abs(est_g.var() - var_g_theory) < 0.05 * var_g_theory,
      f"var Rademacher {est_r.var():.3f} vs Gaussian {est_g.var():.3f}")

### 7.7 Denoising score matching and its minimiser

*Paper:* `EQ-M-022` main paper, `03_foundations.tex:141-147`, `eq:dsm`; `EQ-S1-034` Supplement S1, `03_foundations.tex:354-367`, `eq:dsm + equivalence`; `EQ-S4-038` Supplement S4, `06e_ebm_score.tex:258-268`, `eq:dsm_local and its minimiser`

$$
\mathcal{L}_{\mathrm{DSM}}(\theta;\sigma)=\mathbb{E}_{\mathbf{x}\sim p_{\mathrm{data}},\,\boldsymbol\epsilon\sim\mathcal{N}(0,I)}\Big[\big\|\mathbf{s}_\theta(\mathbf{x}+\sigma\boldsymbol\epsilon)+\boldsymbol\epsilon/\sigma\big\|^2\Big],\qquad\arg\min_{\mathbf{s}}\mathcal{L}_{\mathrm{DSM}}=\nabla\log\big(p_{\mathrm{data}}*\mathcal{N}(0,\sigma^2I)\big)=:\nabla\log p_\sigma
$$

**In plain words.** Add Gaussian noise of size $\sigma$ to a data point and ask the model to point *back* towards the clean point: the target is $-\boldsymbol\epsilon/\sigma$, the direction from the noisy point to the clean one, scaled. No Jacobian is needed. The price is that the best possible model is not the score of the data but the score of the data *blurred* by that noise, $p_\sigma$. With a small $\sigma$ the two are close; with a large $\sigma$ the blurred density is smooth and easy to learn everywhere, even far from the data. Diffusion models use a whole range of $\sigma$.

**Refresher.** Vincent (2011): the expected squared difference between $\mathbf{s}_\theta(\tilde{\mathbf{x}})$ and the *conditional* score $\nabla_{\tilde{\mathbf{x}}}\log q_\sigma(\tilde{\mathbf{x}}\mid\mathbf{x})=-(\tilde{\mathbf{x}}-\mathbf{x})/\sigma^2=-\boldsymbol\epsilon/\sigma$ equals, up to a $\theta$-independent constant, $\mathbb{E}_{p_\sigma}\|\mathbf{s}_\theta-\nabla\log p_\sigma\|^2$, because $\nabla\log p_\sigma(\tilde{\mathbf{x}})=\mathbb{E}[\nabla\log q_\sigma(\tilde{\mathbf{x}}\mid\mathbf{x})\mid\tilde{\mathbf{x}}]$ (the marginal score is the posterior mean of the conditional score; Tweedie's formula). The paper notes that eq:dsm omits the $\tfrac12$ of the Fisher divergence, so $\mathcal{L}_{\mathrm{DSM}}=2\cdot\tfrac12\mathrm{Fisher}(p_\sigma)+C$. The minimiser is $\nabla\log p_\sigma$, not $\nabla\log p_{\mathrm{data}}$: the paper stresses this because it is why a single noise level fails on natural images (the data score is undefined off the manifold) and why many levels are used (Song and Ermon, 2019). The check ports `dsm_equals_esm_on_smoothed_density_up_to_constant` symbolically and adds a torch regression on a two-point dataset where $p_\sigma$ is a two-component mixture with a known score.

**Verify.**

In [ ]:
# Symbolic: data N(mu, s^2), model score -(y - a)/b; DSM equals E_{p_sigma}||s - grad log p_sigma||^2 + const,
# where p_sigma = N(mu, s^2 + sigma^2); and the DSM minimiser is the score of p_sigma.
u, e, w, a, mu = sp.symbols("u e w a mu", real=True)
b, s, sig = sp.symbols("b s sigma", positive=True)
model_score = lambda y: -(y - a) / b
xsym = mu + s * u
dsm = gauss_expect((model_score(xsym + sig * e) + e / sig) ** 2, [u, e])
var_sig = s**2 + sig**2
ysym = mu + sp.sqrt(var_sig) * w                                    # y ~ p_sigma
esm = gauss_expect((model_score(ysym) + (ysym - mu) / var_sig) ** 2, [w])
diff = sp.simplify(dsm - esm)
check("DSM - E_{p_sigma}||s - grad log p_sigma||^2 does not depend on (a, b)", not (diff.free_symbols & {a, b}), f"difference = {diff}")
sol = sp.solve([sp.diff(dsm, a), sp.diff(dsm, b)], [a, b], dict=True)
check("argmin DSM = score of the SMOOTHED density N(mu, s^2 + sigma^2), not of the data",
      any(sp.simplify(d_[a] - mu) == 0 and sp.simplify(d_[b] - var_sig) == 0 for d_ in sol), f"solutions: {sol}")

# Numeric: two-point data {-1, +1}; p_sigma is a two-Gaussian mixture; fit a free-form score by DSM in torch.
sigma = 0.7
data = torch.tensor([-1.0, 1.0])
net = torch.nn.Sequential(torch.nn.Linear(1, 64), torch.nn.Tanh(), torch.nn.Linear(64, 64), torch.nn.Tanh(), torch.nn.Linear(64, 1))
opt = torch.optim.Adam(net.parameters(), 3e-3)
g = torch.Generator().manual_seed(0)
for it in range(1500):
    x0 = data[torch.randint(0, 2, (512,), generator=g)]
    epsn = torch.randn(512, generator=g)
    xt = x0 + sigma * epsn
    loss = ((net(xt[:, None]).squeeze(-1) + epsn / sigma) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
grid = torch.linspace(-2.5, 2.5, 101)
with torch.no_grad():
    fitted = net(grid[:, None]).squeeze(-1).numpy()
gx = grid.numpy()
# exact score of p_sigma = 0.5 N(-1, sigma^2) + 0.5 N(1, sigma^2)
r = np.tanh(gx / sigma**2)                       # posterior mean of x0 given x_t
true_score = -(gx - r) / sigma**2
err = np.sqrt(np.mean((fitted - true_score) ** 2) / np.mean(true_score**2))
check("DSM-trained network approximates grad log p_sigma (relative RMS error < 10%)", err < 0.10, f"relative RMS error {err:.3f}")
check("...which is NOT the data score (data score is undefined off {-1,+1}); e.g. at x=0 the smoothed score is 0",
      abs(true_score[50]) < 1e-6 and abs(fitted[50]) < 0.2 * np.abs(true_score).max(),
      f"true score at 0: {true_score[50]:.1e}; fitted {fitted[50]:.3f} against a score range of {np.abs(true_score).max():.2f}")

## Sampling with a score

Given a score, unadjusted Langevin dynamics turns it into samples. It is exact only in the limit of small steps and long chains; the bias at finite step size is computable in the Gaussian case.

### 7.8 Langevin dynamics

*Paper:* `EQ-S1-035` Supplement S1, `03_foundations.tex:367-372`, `eq:langevin`; `EQ-S4-034` Supplement S4, `06e_ebm_score.tex:45-51`, `eq:ebm_langevin`

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+\frac{\kappa}{2}\,\mathbf{s}_\theta(\mathbf{x}_k)+\sqrt{\kappa}\,\boldsymbol\epsilon_k=\mathbf{x}_k-\frac{\kappa}{2}\,\nabla_{\mathbf{x}}E_\theta(\mathbf{x}_k)+\sqrt{\kappa}\,\boldsymbol\epsilon_k,\qquad\boldsymbol\epsilon_k\sim\mathcal{N}(0,I)
$$

**In plain words.** Take a small step uphill along the score (towards higher probability), then add a little random noise, and repeat. The uphill steps pull the point towards the modes; the noise keeps it from collapsing onto them and lets it explore. With step size $\kappa\to0$ and many steps, the points are distributed as $p_\theta$. For an EBM the uphill direction is minus the energy gradient: Langevin sampling is noisy gradient descent on the energy.

**Refresher.** This is the Euler–Maruyama discretisation of the overdamped Langevin SDE $d\mathbf{x}=\tfrac12\nabla\log p\,dt+d\mathbf{w}$, whose stationary density is $p$ (its Fokker–Planck equation has $p$ as a fixed point; Chapter 8). The *unadjusted* algorithm (no Metropolis correction) is biased at finite $\kappa$: for $p=\mathcal{N}(m,v)$ the chain is an AR(1) process with stationary variance $v/(1-\kappa/(4v))$, which exceeds $v$ and diverges at $\kappa=4v$; the chain converges to $p_\theta$ only as $\kappa\to0$, $k\to\infty$, as the paper says. On real data (Song and Ermon, 2019) two further obstacles arise: the score is undefined off the data manifold and is badly estimated in low-density regions, and chains misweight separated modes; both motivate annealing through noise levels (next entries). The check ports `langevin_stationary_distribution` (Gaussian target, two step sizes, the AR(1) prediction) and adds a double-well energy with a histogram against $e^{-E}/Z$.

**Verify.**

In [ ]:
# (1) Gaussian target N(m, v): ULA's stationary variance is v / (1 - kappa/(4v)) and tends to v as kappa -> 0.
m, v = 2.0, 0.25
rng4 = np.random.default_rng(12)
rows = []
for kappa in (0.05, 0.005):
    x = rng4.normal(size=100_000)
    for _ in range(int(20 / kappa * v) + 200):
        x = x + 0.5 * kappa * (-(x - m) / v) + math.sqrt(kappa) * rng4.normal(size=x.size)
    rows.append((kappa, x.mean(), x.var(), v / (1 - kappa / (4 * v))))
ok = all(abs(mean - m) < 0.01 and abs(var - th) < 0.01 * th for _, mean, var, th in rows) and abs(rows[-1][2] - v) < 0.01
check("ULA on a Gaussian: mean exact, variance = v/(1 - kappa/4v), -> v as kappa -> 0", ok,
      "; ".join(f"kappa={k}: mean {mean:.3f}, var {var:.4f} (AR(1) theory {th:.4f}, target {v})" for k, mean, var, th in rows))

# (2) Double-well energy E(x) = (x^2 - 1)^2 / (2 * 0.15): Langevin histogram against exp(-E)/Z.
temp = 0.15
E = lambda x: (x**2 - 1) ** 2 / (2 * temp)
gradE = lambda x: 2 * x * (x**2 - 1) / temp
xs = np.linspace(-2.5, 2.5, 2001)
pz = np.exp(-E(xs)); Z = np.trapezoid(pz, xs); p_target = pz / Z
kappa, n, K = 0.002, 50_000, 6000
x = rng4.normal(size=n) * 0.3                              # start near the barrier at 0, far from equilibrium
for _ in range(K):
    x = x - 0.5 * kappa * gradE(x) + math.sqrt(kappa) * rng4.normal(size=n)
hist, edges = np.histogram(x, bins=60, range=(-2.5, 2.5), density=True)
centres = 0.5 * (edges[1:] + edges[:-1])
p_at = np.interp(centres, xs, p_target)
tv = 0.5 * np.sum(np.abs(hist - p_at)) * (edges[1] - edges[0])
check("ULA on a double well: histogram matches exp(-E)/Z (total variation < 0.03)", tv < 0.03, f"TV distance {tv:.4f}")
check("both wells are populated with equal mass (symmetric E)", abs((x > 0).mean() - 0.5) < 0.02, f"mass in right well {(x > 0).mean():.3f}")

plt.figure(figsize=(5, 2.6))
plt.bar(centres, hist, width=edges[1] - edges[0], alpha=0.5, label="Langevin samples")
plt.plot(xs, p_target, "k", lw=1.2, label=r"$e^{-E}/Z$")
plt.xlabel("x"); plt.ylabel("density"); plt.title("Unadjusted Langevin dynamics on a double well"); plt.legend(frameon=False)
plt.tight_layout(); plt.show()

### 7.9 A deep EBM in practice: the IGEBM loss and decoupled sampler noise

*Paper:* `EQ-S4-036` Supplement S4, `06e_ebm_score.tex:200-209`, (inline) IGEBM loss and decoupled noise

$$
\mathcal{L}=E_\theta(\mathbf{x})-E_\theta(\mathbf{x}')+\alpha\big(E_\theta(\mathbf{x})^2+E_\theta(\mathbf{x}')^2\big),\qquad \mathbf{x}'\leftarrow\mathbf{x}'-\lambda\,\nabla_{\mathbf{x}}E_\theta(\mathbf{x}')+\omega\,\boldsymbol\epsilon,\quad \lambda\approx10,\ \omega=0.005
$$

**In plain words.** Du and Mordatch (2019) train a convolutional energy with the two-phase rule in its sample form: lower the energy of a data image $\mathbf{x}$, raise that of a model sample $\mathbf{x}'$, and add a penalty $\alpha(E^2+E'^2)$ that stops energies from drifting to huge values. The samples $\mathbf{x}'$ come from a few dozen Langevin-like steps started from a replay buffer. Their sampler uses a gradient step of about $10$ and a separate noise scale of $0.005$, instead of the $\sqrt{\kappa}$ that the Langevin rule prescribes.

**Refresher.** The loss is the per-sample two-phase gradient of EQ 7.2 (the expectation over $\mathbf{x}'$ replaced by a short-run chain) plus an $L_2$ regulariser on the energy magnitude; the chain is not a sampler of $p_\theta$. In the paper's words the noise is *decoupled* from $\sqrt{\kappa}$: a Langevin step with gradient coefficient $\lambda=\kappa/2$ should have noise $\sqrt{\kappa}=\sqrt{2\lambda}$; using $\omega=0.005$ with $\lambda=10$ targets the colder density $\propto e^{-E/T}$ with $T=\omega^2/(2\lambda)$ instead, so the chain seeks low energy more than it samples. The check verifies the algebra of the temperature identification and that the regulariser's gradient is what the formula says, on a Gaussian energy where the cold target is explicit.

**Verify.**

In [ ]:
# A step x' <- x' - lam * grad E + omega * eps is Langevin with step kappa = omega^2 on the energy E/T, T = omega^2 / (2 lam):
# the drift -(kappa/2) grad(E/T) = -lam grad E and the noise sqrt(kappa) = omega both match.
lam, omega = 10.0, 0.005
T_eff = omega**2 / (2 * lam)
check("illustration (arithmetic): IGEBM settings correspond to temperature T = omega^2/(2 lambda)", *close(T_eff, 1.25e-6, 1e-12, f"T = {T_eff:.3e}; "))
# Verify on a quadratic energy E(x) = x^2 / 2 (target variance 1 at T = 1): the chain's stationary variance is T (cold),
# not 1, up to the ULA discretisation factor 1/(1 - kappa/(4 v T)).
lam_s, omega_s = 0.05, 0.1                     # same construction, smaller step so that the chain is stable
kappa_eff, T_s = omega_s**2, omega_s**2 / (2 * lam_s)      # kappa = omega^2 (see above)
x = rng.normal(size=100_000)
for _ in range(2000):
    x = x - lam_s * x + omega_s * rng.normal(size=x.size)
pred = T_s / (1 - kappa_eff / (4 * T_s))        # ULA on N(0, T_s) with step kappa_eff
check("chain with decoupled noise samples the COLDER density (var = T, not 1)", *close(x.var(), pred, 0.02 * pred, f"var {x.var():.4f}, T {T_s}; "))
# Regulariser gradient: d/dE [alpha (E^2 + E'^2)] = 2 alpha E, pulling energies towards zero.
alpha = 1.0
Ev = torch.tensor([3.0, -2.0], requires_grad=True)
reg = alpha * (Ev**2).sum()
reg.backward()
check("regulariser gradient = 2 alpha E (pulls energy magnitudes to zero)", *close(Ev.grad.numpy(), 2 * alpha * Ev.detach().numpy(), 1e-12))

## From score matching to diffusion

Four statements make the bridge: annealing with step sizes proportional to the noise variance, the weighting that equalises the noise levels and turns DSM into the DDPM loss, the division that recovers a score from a noise-prediction network, and the posterior score identity that gives conditional sampling without retraining.

### 7.10 Annealed Langevin, the $\sigma^2$ weighting and the DDPM loss

*Paper:* `EQ-S4-039` Supplement S4, `06e_ebm_score.tex:273-291`, (inline) annealed Langevin; lambda=sigma^2; DSM <-> DDPM; `EQ-M-041` main paper, `06_families.tex:129-134`, (prose) DSM at many noise levels = DDPM loss up to weighting; NCSN/DDPM = VE/VP

$$
\kappa_i\propto\sigma_i^2;\qquad \sum_i\lambda(\sigma_i)\,\mathcal{L}_{\mathrm{DSM}}(\theta;\sigma_i)\ \text{with}\ \lambda(\sigma)=\sigma^2;\qquad\mathbf{s}_\theta(\tilde{\mathbf{x}},\sigma)=-\frac{\boldsymbol\epsilon_\theta(\tilde{\mathbf{x}},\sigma)}{\sigma}\ \Rightarrow\ \mathcal{L}_{\mathrm{DSM}}=\frac{1}{\sigma^2}\,\mathbb{E}\big\|\boldsymbol\epsilon-\boldsymbol\epsilon_\theta\big\|^2
$$

**In plain words.** A score model for natural images is trained at many noise levels $\sigma_1>\dots>\sigma_L$ (Song and Ermon, 2019). Sampling anneals Langevin dynamics from the largest level to the smallest, with a step size proportional to $\sigma_i^2$ so that the step matches the scale of the smoothed density. The losses at different levels have very different sizes (a score is of order $1/\sigma$), so they are weighted by $\sigma^2$ to equalise them. If the network predicts the noise $\boldsymbol\epsilon$ instead of the score, the two are related by a division by $\sigma$, and the weighted DSM loss becomes the plain squared error on the noise: exactly the DDPM training loss.

**Refresher.** Per level, $\|\mathbf{s}_\theta+\boldsymbol\epsilon/\sigma\|^2=\sigma^{-2}\|\boldsymbol\epsilon-\boldsymbol\epsilon_\theta\|^2$ with $\mathbf{s}_\theta=-\boldsymbol\epsilon_\theta/\sigma$, so $\lambda=\sigma^2$ makes every level an unweighted $\boldsymbol\epsilon$-prediction loss; the expected magnitude of the optimal score, $\mathbb{E}\|\nabla\log p_\sigma\|^2\approx D/\sigma^2$ for large $\sigma$, is what $\sigma^2$ cancels. The main paper (EQ-M-041, a wording-fix entry) says DSM at many levels *is* the DDPM loss $\mathcal{L}_{\mathrm{simple}}$ up to the weighting *and the scaling of the data by $\sqrt{\bar\alpha_t}$*: NCSN perturbs $\mathbf{x}+\sigma\boldsymbol\epsilon$ (variance exploding), DDPM perturbs $\sqrt{\bar\alpha_t}\mathbf{x}+\sqrt{1-\bar\alpha_t}\boldsymbol\epsilon$ (variance preserving); the two are the VE and VP SDEs of Chapter 8. The step-size rule $\kappa_i\propto\sigma_i^2$ keeps the signal-to-noise ratio of a Langevin step, $\kappa\|\mathbf{s}\|/(2\sqrt{\kappa})\sim\sqrt{\kappa}/\sigma$, constant across levels. The DDPM-side weight is verified in Chapter 9 (`ddpm_eps_parameterisation_and_vlb_weight`).

**Verify.**

In [ ]:
# (1) Algebra: sigma^2 * ||s + eps/sigma||^2 = ||eps - eps_theta||^2 when s = -eps_theta/sigma (exact, elementwise).
sig = 0.37
epsv = rng.normal(size=(1000, 4)); eps_th = epsv + 0.3 * rng.normal(size=(1000, 4))
s_th = -eps_th / sig
lhs = sig**2 * np.sum((s_th + epsv / sig) ** 2, 1)
rhs = np.sum((epsv - eps_th) ** 2, 1)
check("sigma^2 * DSM integrand = ||eps - eps_theta||^2", *close(lhs, rhs, 1e-10))
# (2) The optimal score has magnitude ~ 1/sigma: E||grad log p_sigma||^2 for 1-D data at {-1,+1}, several sigmas.
def expected_sq_score(sig, n=400_000):
    x0 = rng.choice([-1.0, 1.0], size=n); xt = x0 + sig * rng.normal(size=n)
    sc = -(xt - np.tanh(xt / sig**2)) / sig**2
    return np.mean(sc**2)
sigmas = np.array([0.5, 1.0, 2.0, 4.0, 16.0])
mags = np.array([expected_sq_score(s_) for s_ in sigmas])
ratio = mags * sigmas**2                          # sigma^2 * E||score||^2 -> 1 (= D) for large sigma
check("sigma^2 E||grad log p_sigma||^2 -> D = 1 as sigma grows (why lambda = sigma^2 equalises levels)",
      abs(ratio[-1] - 1) < 0.02 and ratio[-1] > ratio[-2] > ratio[-3],
      f"sigma^2 E||s||^2 = {np.round(ratio, 3)}: not monotone for bimodal data (dip near sigma = 1), approaching 1 like sigma^2/(sigma^2 + 1) once the modes merge")
# (3) Step size kappa_i ∝ sigma_i^2 keeps the Langevin signal-to-noise ratio constant across levels.
snr = lambda kappa, sig: (kappa / 2) * (1 / sig) / math.sqrt(kappa)          # |drift| / noise with |s| ~ 1/sigma
c0 = 0.01
vals = [snr(c0 * s_**2, s_) for s_ in sigmas]
check("illustration (arithmetic): kappa_i = c sigma_i^2 gives a level-independent step signal-to-noise ratio when |s| ~ 1/sigma", *close(vals, vals[0], 1e-12))
# (4) VE vs VP perturbation: same noise, different data scaling.
abar = 0.6
x0 = rng.normal(size=5); epsn = rng.normal(size=5)
ve = x0 + math.sqrt(1 - abar) * epsn; vp = math.sqrt(abar) * x0 + math.sqrt(1 - abar) * epsn
check("illustration (arithmetic): VP perturbation = VE perturbation applied to the data scaled by sqrt(abar)", *close(vp, math.sqrt(abar) * x0 + (ve - x0), 1e-12))

### 7.11 Recovering the score from a noise-prediction network

*Paper:* `EQ-S4-043` Supplement S4, `06e_ebm_score.tex:416-421`, (prose) score from the eps-network by division

$$
\mathbf{s}_\theta(\mathbf{x}_t,t)=-\frac{\boldsymbol\epsilon_\theta(\mathbf{x}_t,t)}{\sqrt{1-\bar\alpha_t}}
$$

**In plain words.** A DDPM network predicts the noise $\boldsymbol\epsilon$ that was added to make $\mathbf{x}_t$. The score of the noisy marginal is minus that prediction divided by the noise's standard deviation, $\sqrt{1-\bar\alpha_t}$. So the repository's diffusion model (`diffusion/model.py`) is also its score model: one network, two readings, and the paper's score-based experiments reuse the diffusion implementation for this reason.

**Refresher.** For $\mathbf{x}_t=\sqrt{\bar\alpha_t}\mathbf{x}_0+\sqrt{1-\bar\alpha_t}\boldsymbol\epsilon$ the conditional score is $\nabla_{\mathbf{x}_t}\log q(\mathbf{x}_t\mid\mathbf{x}_0)=-(\mathbf{x}_t-\sqrt{\bar\alpha_t}\mathbf{x}_0)/(1-\bar\alpha_t)=-\boldsymbol\epsilon/\sqrt{1-\bar\alpha_t}$, and by Tweedie's formula the marginal score is its posterior mean, $\nabla\log q(\mathbf{x}_t)=-\mathbb{E}[\boldsymbol\epsilon\mid\mathbf{x}_t]/\sqrt{1-\bar\alpha_t}$; the DSM/DDPM minimiser $\boldsymbol\epsilon^*_\theta=\mathbb{E}[\boldsymbol\epsilon\mid\mathbf{x}_t]$ gives the identity. It is the VP instance of $\mathbf{s}=-\boldsymbol\epsilon_\theta/\sigma$ with $\sigma_t=\sqrt{1-\bar\alpha_t}$ as the noise standard deviation in $\mathbf{x}_t$'s units. The registry marks the paper's statement as a wording fix: the identity holds for the *optimal* $\boldsymbol\epsilon_\theta$ exactly, and for a trained one it defines the model's score. The check verifies it on Gaussian data, where $\mathbb{E}[\boldsymbol\epsilon\mid\mathbf{x}_t]$ and $\nabla\log q(\mathbf{x}_t)$ are both closed-form.

**Verify.**

In [ ]:
# Gaussian data x0 ~ N(m0, v0); x_t = sqrt(abar) x0 + sqrt(1-abar) eps; the marginal is N(sqrt(abar) m0, abar v0 + 1 - abar).
m0, v0, abar = 1.5, 0.09, 0.3
mt, vt = math.sqrt(abar) * m0, abar * v0 + 1 - abar
xt = np.linspace(-3, 4, 15)
true_score = -(xt - mt) / vt
# optimal eps-prediction E[eps | x_t]: eps = (x_t - sqrt(abar) x0)/sqrt(1-abar), x0 | x_t Gaussian (linear regression)
k = math.sqrt(abar) * v0 / vt                            # Cov(x0, x_t)/Var(x_t)
E_x0_given_xt = m0 + k * (xt - mt)
eps_star = (xt - math.sqrt(abar) * E_x0_given_xt) / math.sqrt(1 - abar)
check("score = -E[eps | x_t] / sqrt(1 - abar) (Gaussian case, closed form)", *close(true_score, -eps_star / math.sqrt(1 - abar), 1e-12))
# Monte-Carlo version of E[eps | x_t] at one point, by kernel regression over simulated pairs
n = 2_000_000
x0s = m0 + math.sqrt(v0) * rng.normal(size=n); e_s = rng.normal(size=n)
xts = math.sqrt(abar) * x0s + math.sqrt(1 - abar) * e_s
x_star = 0.8
w = np.exp(-0.5 * ((xts - x_star) / 0.01) ** 2)
eps_mc = np.sum(w * e_s) / np.sum(w)
check("Monte-Carlo E[eps | x_t] agrees with the closed form", *close(eps_mc, (x_star - math.sqrt(abar) * (m0 + k * (x_star - mt))) / math.sqrt(1 - abar), 0.03))

### 7.12 The posterior score: conditioning without retraining

*Paper:* `EQ-M-042` main paper, `06_families.tex:134-136`, (prose) posterior score; `EQ-S4-042` Supplement S4, `06e_ebm_score.tex:387-389`, (prose) posterior score = prior score + likelihood gradient; 'no retraining'

$$
\nabla_{\mathbf{x}}\log p(\mathbf{x}\mid\mathbf{y})=\nabla_{\mathbf{x}}\log p(\mathbf{x})+\nabla_{\mathbf{x}}\log p(\mathbf{y}\mid\mathbf{x})
$$

**In plain words.** Bayes' rule says $p(\mathbf{x}\mid\mathbf{y})\propto p(\mathbf{x})\,p(\mathbf{y}\mid\mathbf{x})$. Take the log and the gradient in $\mathbf{x}$: the normaliser $p(\mathbf{y})$ does not depend on $\mathbf{x}$ and vanishes, leaving the score of the posterior as the score of the prior plus the gradient of the log-likelihood of the observation. If a model already knows the prior score (an unconditional diffusion model), conditioning on $\mathbf{y}$ (a class, a text prompt, the known pixels in inpainting) only needs the second term added at sampling time.

**Refresher.** Exact for any differentiable prior and likelihood. The subtlety, which the registry's wording fix records, is that a diffusion sampler needs the posterior score of the *noisy* variable $\mathbf{x}_t$, i.e. $\nabla_{\mathbf{x}_t}\log p_t(\mathbf{y}\mid\mathbf{x}_t)$ with $p_t(\mathbf{y}\mid\mathbf{x}_t)=\int p(\mathbf{y}\mid\mathbf{x}_0)p(\mathbf{x}_0\mid\mathbf{x}_t)\,d\mathbf{x}_0$, which is not the clean likelihood: it must be *learned* (a noise-conditional classifier, Dhariwal and Nichol, 2021) or *approximated* (Song et al., 2021; Chung et al., 2023). 'No retraining of the score model' is therefore exact for the prior and approximate for the guidance term. Classifier-free guidance (Chapter 9) replaces the likelihood gradient by a scaled difference of conditional and unconditional scores; `cfg_convention_and_tilt` checks that reading. The check verifies the identity symbolically and on a Gaussian prior with a linear-Gaussian observation, where the posterior is Gaussian in closed form; it also shows that at $t>0$ the clean likelihood's gradient is the wrong term.

**Verify.**

In [ ]:
# Symbolic: log posterior = log prior + log likelihood - log p(y); the last term has zero x-gradient.
xx, yy = sp.symbols("x y", real=True)
m0, v0, vy = sp.symbols("m_0 v_0 v_y", positive=True)
log_prior = -(xx - m0) ** 2 / (2 * v0)
log_lik = -(yy - xx) ** 2 / (2 * vy)
# exact Gaussian posterior N(mp, vp) with vp = 1/(1/v0 + 1/vy), mp = vp (m0/v0 + y/vy)
vp = 1 / (1 / v0 + 1 / vy); mp = vp * (m0 / v0 + yy / vy)
log_post = -(xx - mp) ** 2 / (2 * vp)
lhs = sp.diff(log_post, xx); rhs = sp.diff(log_prior, xx) + sp.diff(log_lik, xx)
check("symbolic: grad_x log p(x|y) = grad_x log p(x) + grad_x log p(y|x)", sp.simplify(lhs - rhs) == 0)
# Numeric on a grid with a non-Gaussian prior (two-component mixture) and a Gaussian likelihood.
xs = np.linspace(-6, 6, 4001)
prior = 0.4 * scipy.stats.norm.pdf(xs, -1.5, 0.6) + 0.6 * scipy.stats.norm.pdf(xs, 1.0, 0.9)
y_obs, sy = 0.3, 1.2
lik = scipy.stats.norm.pdf(y_obs, xs, sy)
post = prior * lik; post /= np.trapezoid(post, xs)
grad = lambda f: np.gradient(np.log(f), xs)
mid = slice(500, 3500)
check("grid: posterior score = prior score + likelihood gradient (mixture prior)",
      *close(grad(post)[mid], (grad(prior) + grad(lik))[mid], 1e-4))
# At noise level t > 0 the needed term is grad log p_t(y | x_t), not the clean likelihood gradient: Gaussian example.
abar, vlik = 0.5, 0.2
# prior x0 ~ N(0,1); x_t = sqrt(abar) x0 + sqrt(1-abar) eps ~ N(0,1); y = x0 + N(0, vlik)
# p_t(y | x_t): x0 | x_t ~ N(sqrt(abar) x_t, 1 - abar), so y | x_t ~ N(sqrt(abar) x_t, 1 - abar + vlik)
xt_pt, y_pt = 0.7, 1.1
grad_noisy_lik = math.sqrt(abar) * (y_pt - math.sqrt(abar) * xt_pt) / (1 - abar + vlik)
grad_clean_lik_at_xt = (y_pt - xt_pt) / vlik
check("at t > 0 the correct guidance term grad log p_t(y|x_t) differs from the clean likelihood gradient",
      abs(grad_noisy_lik - grad_clean_lik_at_xt) > 0.1, f"noisy {grad_noisy_lik:.3f} vs clean {grad_clean_lik_at_xt:.3f}")

## Key takeaways

- An energy-based model defines $p_\theta\propto e^{-E_\theta}$; the normaliser $Z_\theta$ is intractable, but the score $-\nabla E$ does not need it.
- The likelihood gradient is a data term minus a model term; the model term needs samples from the model.
- Score matching fits the score directly; denoising score matching does so by predicting the noise added to data.
- Langevin dynamics samples with only the score, but crosses high-energy barriers slowly; annealing through noise levels fixes this and leads to diffusion.

## Check your understanding

**1. Why does the score $\nabla_{\mathbf x}\log p$ not depend on the normaliser?**

<details><summary>Show the answer</summary>

$\log Z$ is a constant in $\mathbf x$, so its gradient is zero.

</details>

**2. What does a denoising-score-matching network learn: the score of the data or of the noisy data?**

<details><summary>Show the answer</summary>

The score of the data smoothed by the noise; it approaches the data score only as the noise level goes to zero.

</details>

**3. Langevin chains all started in one well rarely reach the other. Which knob helps?**

<details><summary>Show the answer</summary>

Lowering the barrier (or annealing from high noise), or running many more steps; a larger step size also helps but adds discretisation error.

</details>

## Summary of checks

In [ ]:
# Summary of every check in this notebook. This cell raises if any check failed, so that running the
# notebook end to end (locally, on Colab, or in the repository's gate) is itself the verification.
n_ok = sum(ok for _, ok, _ in RESULTS)
print(f"{n_ok}/{len(RESULTS)} checks passed")
for name, ok, detail in RESULTS:
    print(("  ok    " if ok else "  FAIL  ") + name)
failed = [name for name, ok, _ in RESULTS if not ok]
assert not failed, f"failed checks: {failed}"